# ROGII - Modeling Notebook

This notebook combines the simple deterministic baselines and the feature-baseline tree model for the ROGII Wellbore Geology Prediction competition.

Workflow:

1. Load train/test wells and the submission template.
2. Build inference-safe rolling features from `GR`, `TVT_input`, `MD`, and coordinates.
3. Evaluate deterministic baselines: carry-forward, linear trend, damped trend, and blends.
4. Train a residual tree model using held-out-well masked-tail validation.
5. Generate `submission.csv`, using the feature model only if validation beats carry-forward.

The notebook does not use train-only geology-top columns such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, or `BUDA`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

RANDOM_STATE = 42
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT:', DATA_ROOT)
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())

## 1. Load Files

The notebook uses only horizontal well CSVs and `sample_submission.csv`. Typewell alignment is deliberately left for the next modeling stage so we can first measure whether basic rolling features improve on carry-forward.

In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_files = find_files(DATA_ROOT / 'train', '*__horizontal_well.csv')
test_files = find_files(DATA_ROOT / 'test', '*__horizontal_well.csv')
sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]
parsed_ids = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [item[0] for item in parsed_ids]
sample_submission['row_idx'] = [item[1] for item in parsed_ids]

print('train wells:', len(train_files))
print('test wells:', len(test_files))
print('submission rows:', len(sample_submission))
display(sample_submission.head())

## 2. Feature Engineering

Features are designed to be available at inference time. The model never uses train-only geology tops such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, or `BUDA`.

Feature families:

- relative position and measured-depth features;
- per-well centered/normalized coordinates;
- `GR` missingness, interpolation, and rolling statistics;
- carry-forward `TVT_input` features;
- distance from the last known `TVT_input` point;
- recent `TVT_input` slope and local volatility.

The tree model predicts residuals over carry-forward rather than absolute TVT. This makes the baseline safer: if the model cannot improve, the fallback is still carry-forward.

Validation-only columns such as `tail_fraction` are excluded from model features so the same feature list is available when predicting real test wells.

In [ ]:
ROLL_WINDOWS = (25, 101, 301)
FEATURE_COLUMNS = None
NON_FEATURE_COLUMNS = {'well', 'target_tvt', 'target_residual', 'tail_fraction'}


def numeric_col(df, name, default=np.nan):
    col = get_column(df, name)
    if col is None:
        return pd.Series(default, index=df.index, dtype='float64')
    return pd.to_numeric(df[col], errors='coerce').astype('float64')


def tvt_input_series(df):
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    if tvt_input_col is not None:
        return pd.to_numeric(df[tvt_input_col], errors='coerce').astype('float64').reset_index(drop=True)
    if tvt_col is not None:
        return pd.to_numeric(df[tvt_col], errors='coerce').astype('float64').reset_index(drop=True)
    return pd.Series(np.nan, index=range(len(df)), dtype='float64')


def carry_forward_prediction(df):
    y_input = tvt_input_series(df)
    carry = y_input.ffill().bfill()
    if carry.isna().all():
        carry = pd.Series(0.0, index=range(len(df)), dtype='float64')
    return carry.astype('float64')


def add_rolling_features(out, source, prefix):
    for window in ROLL_WINDOWS:
        rolled = source.rolling(window=window, min_periods=1)
        out[f'{prefix}_roll_mean_{window}'] = rolled.mean()
        out[f'{prefix}_roll_std_{window}'] = rolled.std().fillna(0.0)
        out[f'{prefix}_roll_min_{window}'] = rolled.min()
        out[f'{prefix}_roll_max_{window}'] = rolled.max()
    return out


def build_features(df, well):
    n = len(df)
    idx = pd.Series(np.arange(n), dtype='float64')
    denom = max(n - 1, 1)

    md = numeric_col(df, 'MD').reset_index(drop=True)
    x = numeric_col(df, 'X').reset_index(drop=True)
    y = numeric_col(df, 'Y').reset_index(drop=True)
    z = numeric_col(df, 'Z').reset_index(drop=True)
    gr_raw = numeric_col(df, 'GR').reset_index(drop=True)
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)

    gr_interp = gr_raw.interpolate(limit_direction='both').ffill().bfill()
    if gr_interp.isna().all():
        gr_interp = pd.Series(0.0, index=range(n), dtype='float64')

    known = y_input.notna()
    known_idx = pd.Series(np.where(known, idx, np.nan)).ffill().fillna(0.0)
    distance_from_known = (idx - known_idx).clip(lower=0)
    hidden_flag = (~known).astype('int8')

    tvt_diff = carry.diff().fillna(0.0)
    recent_slope = tvt_diff.rolling(window=101, min_periods=1).mean().fillna(0.0)
    recent_volatility = tvt_diff.rolling(window=101, min_periods=1).std().fillna(0.0)

    out = pd.DataFrame({
        'well': well,
        'row_idx': idx,
        'n_rows': float(n),
        'rel_pos': idx / denom,
        'distance_from_known': distance_from_known,
        'distance_from_known_frac': distance_from_known / denom,
        'hidden_flag': hidden_flag,
        'md': md,
        'md_rel': (md - md.min()) / (md.max() - md.min()) if md.notna().sum() > 1 and md.max() != md.min() else idx / denom,
        'x_centered': x - x.mean(),
        'y_centered': y - y.mean(),
        'z_centered': z - z.mean(),
        'gr': gr_raw,
        'gr_interp': gr_interp,
        'gr_missing': gr_raw.isna().astype('int8'),
        'gr_centered': gr_interp - gr_interp.mean(),
        'carry_tvt': carry,
        'tvt_recent_slope': recent_slope,
        'tvt_recent_volatility': recent_volatility,
    })

    out = add_rolling_features(out, gr_interp, 'gr')
    out = add_rolling_features(out, carry, 'carry_tvt')

    numeric_features = [col for col in out.columns if col != 'well']
    out[numeric_features] = out[numeric_features].replace([np.inf, -np.inf], np.nan)
    out[numeric_features] = out[numeric_features].fillna(out[numeric_features].median(numeric_only=True)).fillna(0.0)
    return out


def make_masked_frame(df, tail_fraction):
    tvt_col = get_column(df, 'TVT')
    if tvt_col is None:
        return None, None, None
    y_true = numeric_col(df, 'TVT').reset_index(drop=True)
    eval_start = int(len(df) * (1 - tail_fraction))
    masked = df.copy().reset_index(drop=True)
    masked['TVT_input'] = y_true.copy()
    masked.loc[eval_start:, 'TVT_input'] = np.nan
    return masked, y_true, eval_start


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))) if mask.any() else np.nan


def select_feature_columns(frame):
    return [col for col in frame.columns if col not in NON_FEATURE_COLUMNS]


def align_feature_frame(frame):
    aligned = frame.reindex(columns=FEATURE_COLUMNS, fill_value=0.0).copy()
    aligned = aligned.replace([np.inf, -np.inf], np.nan)
    return aligned.fillna(0.0)


# quick smoke test on one train well
if train_files:
    sample_df = pd.read_csv(train_files[0])
    masked_df, y_true, eval_start = make_masked_frame(sample_df, tail_fraction=0.30)
    sample_features = build_features(masked_df, well_name_from_horizontal_path(train_files[0]))
    print('sample feature shape:', sample_features.shape)
    display(sample_features.head())

## 3. Build Train And Validation Samples

Validation is by held-out wells, not random rows. This is stricter because it asks whether the model generalizes to wells it did not see during training.

To keep runtime reasonable, each simulated hidden suffix is downsampled to a fixed number of rows. The model still sees many wells and multiple hidden-window lengths.

In [ ]:
TAIL_FRACTIONS = (0.20, 0.30, 0.40)
MAX_TRAIN_WELLS = 520
MAX_VALIDATION_WELLS = 160
MAX_ROWS_PER_WELL_FOLD = 900
VALIDATION_WELL_FRACTION = 0.20


def sample_eval_rows(features, y_true, eval_start, max_rows, random_state):
    eval_idx = np.arange(eval_start, len(features))
    if len(eval_idx) > max_rows:
        rng = np.random.default_rng(random_state)
        eval_idx = np.sort(rng.choice(eval_idx, size=max_rows, replace=False))
    sampled = features.iloc[eval_idx].copy()
    sampled['target_tvt'] = y_true.iloc[eval_idx].to_numpy()
    sampled['target_residual'] = sampled['target_tvt'] - sampled['carry_tvt']
    return sampled


def build_modeling_table(files, max_wells, tail_fractions, max_rows_per_fold, seed=RANDOM_STATE):
    frames = []
    for well_number, path in enumerate(files[:max_wells]):
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for frac_number, tail_fraction in enumerate(tail_fractions):
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            features = build_features(masked, well)
            sampled = sample_eval_rows(
                features,
                y_true,
                eval_start,
                max_rows=max_rows_per_fold,
                random_state=seed + 1000 * well_number + frac_number,
            )
            sampled['tail_fraction'] = tail_fraction
            frames.append(sampled)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


rng = np.random.default_rng(RANDOM_STATE)
train_paths = np.array(train_files[:MAX_TRAIN_WELLS], dtype=object)
rng.shuffle(train_paths)
valid_size = max(1, int(len(train_paths) * VALIDATION_WELL_FRACTION))
valid_paths = list(train_paths[:valid_size])
model_train_paths = list(train_paths[valid_size:])

train_table = build_modeling_table(model_train_paths, MAX_TRAIN_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
valid_table = build_modeling_table(valid_paths, MAX_VALIDATION_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)

FEATURE_COLUMNS = select_feature_columns(train_table)

print('model train wells:', len(model_train_paths))
print('validation wells:', len(valid_paths))
print('train table:', train_table.shape)
print('valid table:', valid_table.shape)
print('feature count:', len(FEATURE_COLUMNS))
display(train_table.head())

## 4. Deterministic Baseline Comparison

Before training a tree model, compare inference-safe deterministic baselines on the same held-out validation wells. This keeps the modeling decision grounded: a feature model has to beat carry-forward, not just produce a submission.

In [ ]:
def linear_trend_prediction(df, tail_points=500):
    x = numeric_col(df, 'MD').reset_index(drop=True)
    if x.notna().sum() < 2:
        x = pd.Series(np.arange(len(df), dtype='float64'))
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)
    known = y_input.notna() & x.notna()
    if known.sum() < 2:
        return carry

    x_known = x[known].to_numpy()
    y_known = y_input[known].to_numpy()
    n_tail = min(tail_points, len(x_known))
    x_tail = x_known[-n_tail:]
    y_tail = y_known[-n_tail:]
    x_anchor = x_tail[-1]
    y_anchor = y_tail[-1]
    if np.nanstd(x_tail) == 0:
        return carry

    slope = np.polyfit(x_tail - x_anchor, y_tail - y_anchor, 1)[0]
    pred = y_anchor + slope * (x.to_numpy() - x_anchor)
    pred = pd.Series(pred, index=range(len(df)), dtype='float64')
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.ffill().bfill().astype('float64')


def damped_trend_prediction(df, damp=0.35):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df)
    y_input = tvt_input_series(df)
    pred = carry + damp * (trend - carry)
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.astype('float64')


def blended_prediction(df, weight=0.25):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df)
    y_input = tvt_input_series(df)
    pred = (1 - weight) * carry + weight * trend
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.astype('float64')


def deterministic_prediction(df, model_name):
    if model_name == 'carry_forward':
        return carry_forward_prediction(df)
    if model_name == 'linear_trend':
        return linear_trend_prediction(df)
    if model_name == 'damped_trend_035':
        return damped_trend_prediction(df, damp=0.35)
    if model_name == 'blend_025':
        return blended_prediction(df, weight=0.25)
    if model_name == 'blend_050':
        return blended_prediction(df, weight=0.50)
    raise ValueError(model_name)


def evaluate_deterministic_baselines(paths, tail_fractions):
    model_names = ['carry_forward', 'blend_025', 'damped_trend_035', 'blend_050', 'linear_trend']
    rows = []
    for path in paths:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for tail_fraction in tail_fractions:
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            for model_name in model_names:
                pred = deterministic_prediction(masked, model_name)
                rows.append({
                    'well': well,
                    'tail_fraction': tail_fraction,
                    'model': model_name,
                    'rmse': rmse(y_true.iloc[eval_start:], pred.iloc[eval_start:]),
                })
    return pd.DataFrame(rows)


simple_validation = evaluate_deterministic_baselines(valid_paths, TAIL_FRACTIONS)
simple_summary = (
    simple_validation.groupby('model')['rmse']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean')
)
display(simple_summary)

deterministic_winner = simple_summary.index[0]
print('best deterministic baseline:', deterministic_winner)

## 5. Train Feature Baseline

The feature model predicts residuals over carry-forward. Validation reports both the raw carry-forward score and the feature-tree score.

In [ ]:
def fit_feature_model(train_table):
    X = align_feature_frame(train_table)
    y = train_table['target_residual']
    try:
        model = HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=31,
            min_samples_leaf=40,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )
        model.fit(X, y)
        model_name = 'HistGradientBoostingRegressor'
    except Exception as exc:
        print('HistGradientBoostingRegressor failed; falling back to RandomForestRegressor:', repr(exc))
        model = RandomForestRegressor(
            n_estimators=160,
            max_depth=10,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X, y)
        model_name = 'RandomForestRegressor'
    return model, model_name


model, model_name = fit_feature_model(train_table)
print('model:', model_name)

valid_pred_residual = model.predict(align_feature_frame(valid_table))
valid_tree_pred = valid_table['carry_tvt'].to_numpy() + valid_pred_residual
carry_rmse = rmse(valid_table['target_tvt'], valid_table['carry_tvt'])
tree_rmse = rmse(valid_table['target_tvt'], valid_tree_pred)

print('carry_forward validation RMSE:', carry_rmse)
print('feature_tree validation RMSE:', tree_rmse)
print('delta RMSE:', tree_rmse - carry_rmse)

valid_scored = valid_table[['well', 'tail_fraction', 'target_tvt', 'carry_tvt']].copy()
valid_scored['feature_tree'] = valid_tree_pred
valid_summary = (
    valid_scored
    .groupby(['tail_fraction'])
    .apply(lambda x: pd.Series({
        'carry_rmse': rmse(x['target_tvt'], x['carry_tvt']),
        'feature_tree_rmse': rmse(x['target_tvt'], x['feature_tree']),
        'rows': len(x),
        'wells': x['well'].nunique(),
    }))
    .reset_index()
)
valid_summary['delta'] = valid_summary['feature_tree_rmse'] - valid_summary['carry_rmse']
display(valid_summary)

use_tree_model = bool(tree_rmse < carry_rmse)
print('use_tree_model_for_submission:', use_tree_model)

## 6. Refit And Generate Submission

If validation improves, refit the feature model on simulated tails from more training wells and use it for the test hidden rows. If validation does not improve, fall back to carry-forward. This makes the notebook safe to submit while still testing the feature baseline.

In [ ]:
if use_tree_model:
    refit_table = build_modeling_table(train_files, max_wells=min(len(train_files), 720), tail_fractions=TAIL_FRACTIONS, max_rows_per_fold=MAX_ROWS_PER_WELL_FOLD)
    FEATURE_COLUMNS = select_feature_columns(refit_table)
    model, model_name = fit_feature_model(refit_table)
    print('refit model:', model_name)
    print('refit table:', refit_table.shape)
else:
    print('Validation did not beat carry-forward. Submission will use carry-forward fallback.')


def predict_test_well(df, well):
    features = build_features(df.reset_index(drop=True), well)
    carry = features['carry_tvt'].to_numpy()
    if use_tree_model:
        residual = model.predict(align_feature_frame(features))
        pred = carry + residual
        known = tvt_input_series(df).notna().to_numpy()
        pred[known] = tvt_input_series(df).to_numpy()[known]
        return pd.Series(pred, index=range(len(df)), dtype='float64')
    return pd.Series(carry, index=range(len(df)), dtype='float64')


test_lookup = {well_name_from_horizontal_path(path): path for path in test_files}
well_predictions = {}
for well, path in test_lookup.items():
    df = pd.read_csv(path)
    well_predictions[well] = predict_test_well(df, well).reset_index(drop=True)

fallback = 0.0
non_empty = [pred.dropna().to_numpy() for pred in well_predictions.values() if pred.dropna().size]
if non_empty:
    fallback = float(np.nanmedian(np.concatenate(non_empty)))

submission = sample_submission[[id_col]].copy()
values = []
missing_wells = set()
for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred = well_predictions.get(well)
    if pred is None or len(pred) == 0:
        missing_wells.add(well)
        values.append(fallback)
    elif 0 <= row_idx < len(pred):
        values.append(float(pred.iloc[row_idx]))
    else:
        values.append(float(pred.iloc[-1]))

submission[target_col] = values
submission.to_csv(SUBMISSION_PATH, index=False)

print('wrote:', SUBMISSION_PATH)
print('selected submission model:', 'feature_tree' if use_tree_model else 'carry_forward')
print('rows:', len(submission))
print('missing wells:', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())

## 7. Readout

Use the validation delta as the submission decision:

- if `feature_tree_rmse < carry_rmse`, the feature baseline earned a submission;
- if not, the notebook intentionally falls back to carry-forward.

A feature tree that fails here is still useful: it tells us that rolling features alone are insufficient, and the next step should be typewell alignment features rather than more model complexity.